# Statistical Inference for Quantitative Finance

A comprehensive guide to the statistical inference concepts most frequently tested in quant finance interviews. This notebook covers estimation theory, hypothesis testing, regression, Bayesian methods, time series, and Monte Carlo simulation --- all with a finance lens.

---

## 1. Descriptive Statistics

Descriptive statistics summarize the key features of a dataset. In finance, these summaries are the first step in understanding return distributions, risk profiles, and data quality.

### 1.1 Sample Mean, Variance, and Standard Deviation

Given a sample $x_1, x_2, \ldots, x_n$, the fundamental summary statistics are:

**Sample Mean:**

$$\bar{x} = \frac{1}{n} \sum_{i=1}^{n} x_i$$

**Sample Variance** (unbiased):

$$s^2 = \frac{1}{n-1} \sum_{i=1}^{n} (x_i - \bar{x})^2$$

We divide by $n-1$ (Bessel's correction) rather than $n$ to obtain an unbiased estimator of the population variance $\sigma^2$. The intuition: the sample mean $\bar{x}$ is closer to the data than the true mean $\mu$, so dividing by $n$ would systematically underestimate spread.

**Sample Standard Deviation:**

$$s = \sqrt{s^2} = \sqrt{\frac{1}{n-1} \sum_{i=1}^{n}(x_i - \bar{x})^2}$$

Note that while $s^2$ is an unbiased estimator of $\sigma^2$, $s$ is a *biased* estimator of $\sigma$ (due to the concavity of the square root function and Jensen's inequality).

> **Interview Tip:** Be prepared to explain *why* we divide by $n-1$ instead of $n$. The key argument: $\bar{x}$ is estimated from the data, consuming one degree of freedom. Equivalently, the deviations $x_i - \bar{x}$ satisfy $\sum_i(x_i - \bar{x}) = 0$, so only $n-1$ of them are free.

### 1.2 Skewness and Kurtosis

Higher moments capture the *shape* of a distribution beyond its center and spread. These are critical in finance because return distributions are rarely Gaussian.

**Skewness** (third standardized moment) measures asymmetry:

$$\text{Skew}(X) = E\left[\left(\frac{X - \mu}{\sigma}\right)^3\right] = \frac{\mu_3}{\sigma^3}$$

Sample skewness:

$$g_1 = \frac{\frac{1}{n}\sum_{i=1}^{n}(x_i - \bar{x})^3}{\left(\frac{1}{n}\sum_{i=1}^{n}(x_i - \bar{x})^2\right)^{3/2}}$$

- $\text{Skew} > 0$: right tail is heavier (e.g., some commodity returns)
- $\text{Skew} < 0$: left tail is heavier (e.g., equity index returns)
- $\text{Skew} = 0$: symmetric (e.g., normal distribution)

**Kurtosis** (fourth standardized moment) measures tail heaviness:

$$\text{Kurt}(X) = E\left[\left(\frac{X - \mu}{\sigma}\right)^4\right] = \frac{\mu_4}{\sigma^4}$$

**Excess Kurtosis** subtracts the normal distribution's kurtosis of 3:

$$\text{Excess Kurt}(X) = \text{Kurt}(X) - 3$$

- Excess kurtosis $> 0$: **leptokurtic** (fat tails, more extreme events than normal)
- Excess kurtosis $= 0$: **mesokurtic** (normal-like tails)
- Excess kurtosis $< 0$: **platykurtic** (thin tails, fewer extreme events)

> **Interview Tip:** Equity returns almost universally exhibit *negative skewness* and *positive excess kurtosis*. This means crashes are more likely than a normal model predicts, and large moves in general are more frequent. This is why risk models that assume normality underestimate tail risk --- a lesson painfully learned in 2008.

### 1.3 Order Statistics, Quantiles, and Percentiles

**Order Statistics:** Sorting the sample gives $x_{(1)} \leq x_{(2)} \leq \cdots \leq x_{(n)}$. The $k$-th order statistic is $x_{(k)}$.

**Quantile Function (Inverse CDF):**

$$Q(p) = F^{-1}(p) = \inf\{x : F(x) \geq p\}, \quad p \in (0,1)$$

Key quantiles:

| Quantile | Name | Finance Application |
|----------|------|--------------------|
| $Q(0.01)$ | 1st percentile | VaR at 99% confidence |
| $Q(0.05)$ | 5th percentile | VaR at 95% confidence |
| $Q(0.25)$ | First quartile | Interquartile range |
| $Q(0.50)$ | Median | Robust central tendency |

**Value at Risk (VaR)** is directly a quantile:

$$\text{VaR}_{\alpha} = -Q(1 - \alpha) = -F^{-1}(1 - \alpha)$$

For a normal distribution with mean $\mu$ and standard deviation $\sigma$:

$$\text{VaR}_{\alpha} = -(\mu + \sigma \cdot z_{1-\alpha})$$

where $z_{1-\alpha}$ is the $(1-\alpha)$-quantile of the standard normal.

> **Interview Tip:** Know that VaR at the 99% level uses $z_{0.01} \approx -2.326$. At 95%, $z_{0.05} \approx -1.645$. These numbers come up constantly.

---

## 2. Estimation Theory

Estimation theory formalizes how we learn about unknown parameters from data. Understanding the properties of estimators is fundamental for any quantitative role.

### 2.1 Point Estimation: Key Properties

An **estimator** $\hat{\theta}_n = T(X_1, \ldots, X_n)$ is a function of the data used to estimate an unknown parameter $\theta$.

**Bias:**

$$\text{Bias}(\hat{\theta}) = E[\hat{\theta}] - \theta$$

An estimator is **unbiased** if $E[\hat{\theta}] = \theta$ for all $\theta$.

**Consistency:** An estimator is **consistent** if

$$\hat{\theta}_n \xrightarrow{p} \theta \quad \text{as } n \to \infty$$

That is, $P(|\hat{\theta}_n - \theta| > \epsilon) \to 0$ for any $\epsilon > 0$.

A sufficient condition: if $\text{Bias}(\hat{\theta}_n) \to 0$ and $\text{Var}(\hat{\theta}_n) \to 0$ as $n \to \infty$, then $\hat{\theta}_n$ is consistent.

**Efficiency:** Among all unbiased estimators, the **efficient** estimator has the smallest variance. The **relative efficiency** of two unbiased estimators is:

$$e(\hat{\theta}_1, \hat{\theta}_2) = \frac{\text{Var}(\hat{\theta}_2)}{\text{Var}(\hat{\theta}_1)}$$

**Mean Squared Error (MSE)** combines bias and variance:

$$\text{MSE}(\hat{\theta}) = E[(\hat{\theta} - \theta)^2] = \text{Var}(\hat{\theta}) + [\text{Bias}(\hat{\theta})]^2$$

This **bias-variance tradeoff** is central to all of statistics and machine learning: a slightly biased estimator can have lower MSE than an unbiased one if it has substantially lower variance.

> **Interview Tip:** A classic question: "Is the sample standard deviation $s$ an unbiased estimator of $\sigma$?" No --- $s^2$ is unbiased for $\sigma^2$, but $s = \sqrt{s^2}$ is biased for $\sigma$ by Jensen's inequality. However, $s$ is consistent for $\sigma$.

### 2.2 Maximum Likelihood Estimation (MLE)

MLE is the most widely used estimation method. Given iid observations $x_1, \ldots, x_n$ from density $f(x|\theta)$:

**Likelihood Function:**

$$L(\theta) = \prod_{i=1}^{n} f(x_i | \theta)$$

**Log-Likelihood** (easier to work with):

$$\ell(\theta) = \sum_{i=1}^{n} \ln f(x_i | \theta)$$

**MLE:**

$$\hat{\theta}_{\text{MLE}} = \arg\max_{\theta} \ell(\theta)$$

Found by solving the **score equation**:

$$\frac{\partial \ell}{\partial \theta} = 0$$

#### Properties of the MLE (under regularity conditions):

1. **Consistency:** $\hat{\theta}_{\text{MLE}} \xrightarrow{p} \theta_0$ (converges to the true parameter)
2. **Asymptotic normality:** $\sqrt{n}(\hat{\theta}_{\text{MLE}} - \theta_0) \xrightarrow{d} N(0, I(\theta_0)^{-1})$
3. **Asymptotic efficiency:** achieves the Cramer-Rao lower bound asymptotically
4. **Invariance:** if $\hat{\theta}$ is the MLE of $\theta$, then $g(\hat{\theta})$ is the MLE of $g(\theta)$

#### Example: MLE for Normal Distribution

For $X_1, \ldots, X_n \sim N(\mu, \sigma^2)$:

$$\ell(\mu, \sigma^2) = -\frac{n}{2}\ln(2\pi) - \frac{n}{2}\ln(\sigma^2) - \frac{1}{2\sigma^2}\sum_{i=1}^{n}(x_i - \mu)^2$$

Setting partial derivatives to zero:

$$\hat{\mu}_{\text{MLE}} = \bar{x} = \frac{1}{n}\sum_{i=1}^{n} x_i$$

$$\hat{\sigma}^2_{\text{MLE}} = \frac{1}{n}\sum_{i=1}^{n}(x_i - \bar{x})^2$$

Note: $\hat{\sigma}^2_{\text{MLE}}$ divides by $n$, not $n-1$, so it is biased (but consistent and asymptotically efficient).

> **Interview Tip:** MLE derivations for normal, exponential, and Poisson distributions are extremely common interview questions. Practice writing out the log-likelihood, taking derivatives, and solving. For the exponential distribution $f(x|\lambda) = \lambda e^{-\lambda x}$, the MLE is $\hat{\lambda} = 1/\bar{x}$.

### 2.3 Method of Moments (MoM)

The Method of Moments equates population moments to sample moments and solves for the parameters.

The $k$-th **population moment** is $\mu_k' = E[X^k]$ and the $k$-th **sample moment** is $m_k' = \frac{1}{n}\sum_{i=1}^n x_i^k$.

If the distribution has $p$ parameters, set up $p$ equations:

$$\mu_k'(\theta_1, \ldots, \theta_p) = m_k', \quad k = 1, \ldots, p$$

**Example:** For $X \sim N(\mu, \sigma^2)$:
- First moment: $E[X] = \mu \Rightarrow \hat{\mu} = \bar{x}$
- Second moment: $E[X^2] = \mu^2 + \sigma^2 \Rightarrow \hat{\sigma}^2 = m_2' - \bar{x}^2 = \frac{1}{n}\sum(x_i - \bar{x})^2$

MoM estimators are generally consistent but less efficient than MLE. Their advantage is simplicity --- no optimization required.

### 2.4 Fisher Information and the Cramer-Rao Lower Bound

**Fisher Information** quantifies how much information a sample carries about a parameter:

$$I(\theta) = -E\left[\frac{\partial^2}{\partial \theta^2} \ln f(X|\theta)\right] = E\left[\left(\frac{\partial}{\partial \theta}\ln f(X|\theta)\right)^2\right]$$

The two expressions are equal under regularity conditions. For $n$ iid observations, the total Fisher information is $nI(\theta)$.

**Cramer-Rao Lower Bound (CRLB):**

For any unbiased estimator $\hat{\theta}$:

$$\text{Var}(\hat{\theta}) \geq \frac{1}{nI(\theta)}$$

An unbiased estimator achieving this bound is called **efficient** or a **minimum variance unbiased estimator (MVUE)**.

**Example:** For $X \sim N(\mu, \sigma^2)$ with $\sigma^2$ known:

$$I(\mu) = \frac{1}{\sigma^2}, \quad \text{CRLB} = \frac{\sigma^2}{n}$$

Since $\text{Var}(\bar{X}) = \sigma^2/n$, the sample mean achieves the CRLB --- it is efficient for estimating $\mu$.

> **Interview Tip:** If asked "What is the best unbiased estimator of the mean of a normal distribution?", the answer is $\bar{X}$, and you can prove it by showing it achieves the CRLB.

---

## 3. Confidence Intervals

A **confidence interval** provides a range of plausible values for an unknown parameter. A $100(1-\alpha)\%$ confidence interval $[L, U]$ satisfies:

$$P(L \leq \theta \leq U) = 1 - \alpha$$

**Interpretation:** If we repeated the experiment many times and computed an interval each time, approximately $100(1-\alpha)\%$ of those intervals would contain the true parameter $\theta$. The interval is random; $\theta$ is fixed.

### 3.1 Normal-Based Confidence Intervals

For a population mean $\mu$ with **known** variance $\sigma^2$:

$$\bar{X} \pm z_{\alpha/2} \cdot \frac{\sigma}{\sqrt{n}}$$

where $z_{\alpha/2}$ is the upper $\alpha/2$ quantile of $N(0,1)$.

| Confidence Level | $z_{\alpha/2}$ |
|:---:|:---:|
| 90% | 1.645 |
| 95% | 1.960 |
| 99% | 2.576 |

**Margin of error** $= z_{\alpha/2} \cdot \sigma / \sqrt{n}$. Note the $\sqrt{n}$ in the denominator: to halve the margin of error, you need four times the sample size.

### 3.2 The $t$-Distribution and When to Use It

When the population variance $\sigma^2$ is **unknown** (the usual case), we replace it with $s^2$ and use the $t$-distribution:

$$T = \frac{\bar{X} - \mu}{s / \sqrt{n}} \sim t_{n-1}$$

The $100(1-\alpha)\%$ confidence interval for $\mu$:

$$\bar{X} \pm t_{n-1, \alpha/2} \cdot \frac{s}{\sqrt{n}}$$

**Key properties of the $t$-distribution:**
- Heavier tails than the standard normal
- Parameterized by degrees of freedom $\nu = n - 1$
- As $\nu \to \infty$, $t_\nu \to N(0,1)$
- For $\nu > 2$: $\text{Var}(T) = \nu/(\nu - 2) > 1$

**When to use $z$ vs $t$:**
- $\sigma$ known $\Rightarrow$ use $z$
- $\sigma$ unknown, data normal (or $n$ large) $\Rightarrow$ use $t$
- In practice, always use $t$; it reduces to $z$ for large $n$

> **Interview Tip:** A common trap: "You have 25 daily returns and want a confidence interval for the mean. Do you use $z$ or $t$?" Always $t$ --- you never know $\sigma$ in practice. With $n = 25$, $t_{24, 0.025} \approx 2.064$, which is noticeably wider than $z_{0.025} = 1.96$.

### 3.3 Chi-Squared Interval for Variance

For a normal population, the sampling distribution of the variance is:

$$\frac{(n-1)s^2}{\sigma^2} \sim \chi^2_{n-1}$$

A $100(1-\alpha)\%$ confidence interval for $\sigma^2$:

$$\left[\frac{(n-1)s^2}{\chi^2_{n-1, \alpha/2}}, \quad \frac{(n-1)s^2}{\chi^2_{n-1, 1-\alpha/2}}\right]$$

This interval is **not symmetric** around $s^2$ because the $\chi^2$ distribution is right-skewed.

> **Interview Tip:** Confidence intervals for volatility ($\sigma$) are important in options pricing. Take square roots of the endpoints above to get a CI for $\sigma$. Note how wide these intervals can be for small $n$ --- estimating volatility precisely requires a lot of data.

### 3.4 Bootstrap Methods

The **bootstrap** is a resampling method that estimates the sampling distribution of a statistic without parametric assumptions.

**Algorithm (Nonparametric Bootstrap):**

1. From the original sample of size $n$, draw $B$ bootstrap samples of size $n$ **with replacement**
2. Compute the statistic $\hat{\theta}^{*}_b$ for each bootstrap sample $b = 1, \ldots, B$
3. Use the distribution of $\hat{\theta}^{*}_1, \ldots, \hat{\theta}^{*}_B$ to estimate standard errors and confidence intervals

**Bootstrap Standard Error:**

$$\widehat{\text{SE}}_{\text{boot}} = \sqrt{\frac{1}{B-1}\sum_{b=1}^{B}(\hat{\theta}^{*}_b - \bar{\hat{\theta}}^{*})^2}$$

**Bootstrap Percentile CI:** Use the $\alpha/2$ and $1-\alpha/2$ quantiles of the bootstrap distribution:

$$\left[\hat{\theta}^{*}_{(\alpha/2)}, \quad \hat{\theta}^{*}_{(1-\alpha/2)}\right]$$

**Advantages:**
- No distributional assumptions required
- Works for complex statistics (Sharpe ratio, VaR, etc.) where analytic formulas are unavailable
- Automatically captures skewness in the sampling distribution

> **Interview Tip:** "How would you construct a confidence interval for the Sharpe ratio?" The bootstrap is the cleanest answer. There is no simple closed-form CI for the Sharpe ratio because it is a ratio of estimators.

---

## 4. Hypothesis Testing

Hypothesis testing provides a formal framework for making decisions from data. The logic: assume a default state of the world (null hypothesis), and ask whether the data provide sufficient evidence to reject it.

### 4.1 Framework: Null and Alternative Hypotheses

- **Null hypothesis** $H_0$: the default or status quo (e.g., "the strategy has zero alpha")
- **Alternative hypothesis** $H_1$ (or $H_a$): what we want to establish (e.g., "the strategy has positive alpha")

**Types of tests:**
- Two-sided: $H_0: \theta = \theta_0$ vs $H_1: \theta \neq \theta_0$
- One-sided (right): $H_0: \theta \leq \theta_0$ vs $H_1: \theta > \theta_0$
- One-sided (left): $H_0: \theta \geq \theta_0$ vs $H_1: \theta < \theta_0$

### 4.2 Type I and Type II Errors

|  | $H_0$ true | $H_0$ false |
|--|-----------|------------|
| **Reject $H_0$** | Type I error ($\alpha$) | Correct (Power = $1-\beta$) |
| **Fail to reject $H_0$** | Correct | Type II error ($\beta$) |

- **Type I error rate** ($\alpha$): probability of rejecting $H_0$ when it is true (false positive)
- **Type II error rate** ($\beta$): probability of failing to reject $H_0$ when it is false (false negative)
- **Significance level**: the chosen maximum Type I error rate, typically $\alpha = 0.05$
- **Power**: $1 - \beta$ = probability of correctly rejecting a false $H_0$

There is a fundamental tradeoff: reducing $\alpha$ (fewer false positives) increases $\beta$ (more false negatives) for a fixed sample size. The only way to reduce both is to increase $n$.

### 4.3 p-Values and Significance

The **p-value** is the probability, under $H_0$, of observing a test statistic at least as extreme as the one actually observed:

$$p = P(T \geq t_{\text{obs}} \mid H_0) \quad \text{(one-sided)}$$

$$p = P(|T| \geq |t_{\text{obs}}| \mid H_0) \quad \text{(two-sided)}$$

**Decision rule:** Reject $H_0$ if $p \leq \alpha$.

**Important:** A p-value is NOT the probability that $H_0$ is true. It is the probability of the data (or more extreme) given $H_0$. This is a frequent source of misinterpretation.

> **Interview Tip:** "If the p-value is 0.03, what is the probability that $H_0$ is true?" The answer is: "We cannot determine that from the p-value alone. The p-value tells us $P(\text{data} \mid H_0) = 0.03$, not $P(H_0 \mid \text{data})$. For the latter, we need Bayes' theorem and a prior."

### 4.4 Power of a Test

The **power function** maps the true parameter value to the probability of rejection:

$$\pi(\theta) = P(\text{reject } H_0 \mid \theta)$$

For a one-sided $z$-test of $H_0: \mu = \mu_0$ vs $H_1: \mu > \mu_0$ at level $\alpha$:

$$\text{Power}(\mu_1) = P\left(Z > z_{\alpha} \mid \mu = \mu_1\right) = 1 - \Phi\left(z_{\alpha} - \frac{\mu_1 - \mu_0}{\sigma/\sqrt{n}}\right)$$

Power increases when:
- The true effect size $|\mu_1 - \mu_0|$ is larger
- The sample size $n$ increases
- The noise $\sigma$ decreases
- The significance level $\alpha$ increases (but this increases Type I error)

**Required sample size** for a desired power $1-\beta$ (one-sided test):

$$n = \left(\frac{(z_{\alpha} + z_{\beta})\sigma}{\mu_1 - \mu_0}\right)^2$$

### 4.5 Common Tests

#### $z$-test (known variance)

Test statistic for $H_0: \mu = \mu_0$:

$$z = \frac{\bar{x} - \mu_0}{\sigma / \sqrt{n}} \sim N(0,1) \text{ under } H_0$$

#### $t$-test (unknown variance)

**One-sample:** $t = \frac{\bar{x} - \mu_0}{s/\sqrt{n}} \sim t_{n-1}$

**Two-sample (equal variances):**

$$t = \frac{\bar{x}_1 - \bar{x}_2}{s_p\sqrt{1/n_1 + 1/n_2}}, \quad s_p^2 = \frac{(n_1-1)s_1^2 + (n_2-1)s_2^2}{n_1 + n_2 - 2}$$

**Two-sample (unequal variances, Welch's test):**

$$t = \frac{\bar{x}_1 - \bar{x}_2}{\sqrt{s_1^2/n_1 + s_2^2/n_2}}$$

with approximate degrees of freedom given by the Welch-Satterthwaite formula.

#### Chi-squared test for variance

Test $H_0: \sigma^2 = \sigma_0^2$:

$$\chi^2 = \frac{(n-1)s^2}{\sigma_0^2} \sim \chi^2_{n-1} \text{ under } H_0$$

#### $F$-test for comparing two variances

Test $H_0: \sigma_1^2 = \sigma_2^2$:

$$F = \frac{s_1^2}{s_2^2} \sim F_{n_1 - 1, n_2 - 1} \text{ under } H_0$$

> **Interview Tip:** When comparing the volatilities of two assets or strategies, the $F$-test is the natural choice. However, it is sensitive to non-normality, so the bootstrap or Levene's test may be more appropriate with financial data.

### 4.6 Multiple Testing Correction

When conducting $m$ simultaneous tests, the probability of at least one false positive (Type I error) grows rapidly. This is the **multiple comparisons problem**.

If each test is independent at level $\alpha$:

$$P(\text{at least one false positive}) = 1 - (1-\alpha)^m$$

For $m = 20$ tests at $\alpha = 0.05$: $P \approx 0.64$ --- nearly two-thirds of the time you get a false positive!

**Bonferroni Correction:** Reject individual test $i$ only if $p_i \leq \alpha/m$. This controls the **family-wise error rate (FWER)** at level $\alpha$.

**Benjamini-Hochberg (BH) Procedure** (controls **False Discovery Rate, FDR**):
1. Order p-values: $p_{(1)} \leq p_{(2)} \leq \cdots \leq p_{(m)}$
2. Find the largest $k$ such that $p_{(k)} \leq \frac{k}{m}\alpha$
3. Reject all hypotheses $H_{(1)}, \ldots, H_{(k)}$

FDR control is less conservative than FWER and is more appropriate when testing many hypotheses.

> **Interview Tip:** This is **critical** in quantitative finance. When backtesting hundreds of trading strategies, many will appear profitable by chance. If you test 100 strategies, about 5 will have $p < 0.05$ even if none have true alpha. Always correct for multiple testing. The paper *"...and the Cross-Section of Expected Returns"* (Harvey, Liu, Zhu, 2016) argues that a $t$-statistic threshold of about 3.0 (instead of 2.0) should be used for new factor discoveries.

---

## 5. Regression and Least Squares

Regression is the workhorse of empirical finance. It relates a response variable to explanatory variables, forming the basis for factor models, risk attribution, and forecasting.

### 5.1 Simple Linear Regression

The model:

$$Y_i = \beta_0 + \beta_1 X_i + \epsilon_i, \quad \epsilon_i \sim N(0, \sigma^2), \quad i = 1, \ldots, n$$

**OLS Estimators** minimize $\sum_{i=1}^n (Y_i - \beta_0 - \beta_1 X_i)^2$:

$$\hat{\beta}_1 = \frac{\sum_{i=1}^n (X_i - \bar{X})(Y_i - \bar{Y})}{\sum_{i=1}^n (X_i - \bar{X})^2} = \frac{S_{XY}}{S_{XX}}$$

$$\hat{\beta}_0 = \bar{Y} - \hat{\beta}_1 \bar{X}$$

**Finance application:** In the **CAPM regression** $R_i - R_f = \alpha_i + \beta_i(R_M - R_f) + \epsilon_i$:
- $\hat{\beta}_i$ is the asset's **beta** (systematic risk)
- $\hat{\alpha}_i$ is **Jensen's alpha** (abnormal return after adjusting for market risk)
- Testing $H_0: \alpha = 0$ is testing whether the asset earns excess risk-adjusted returns

### 5.2 Multiple Regression and the OLS Estimator

In matrix form with $p$ predictors:

$$\mathbf{Y} = \mathbf{X}\boldsymbol{\beta} + \boldsymbol{\epsilon}$$

where $\mathbf{Y}$ is $n \times 1$, $\mathbf{X}$ is $n \times (p+1)$ (including intercept column), $\boldsymbol{\beta}$ is $(p+1) \times 1$.

**OLS Estimator:**

$$\hat{\boldsymbol{\beta}} = (\mathbf{X}'\mathbf{X})^{-1}\mathbf{X}'\mathbf{Y}$$

**Properties under the classical assumptions** (linearity, exogeneity, no multicollinearity, homoscedasticity, normality):

$$E[\hat{\boldsymbol{\beta}}] = \boldsymbol{\beta} \quad \text{(unbiased)}$$

$$\text{Cov}(\hat{\boldsymbol{\beta}}) = \sigma^2 (\mathbf{X}'\mathbf{X})^{-1}$$

The unbiased estimator of $\sigma^2$:

$$\hat{\sigma}^2 = \frac{\text{RSS}}{n - p - 1} = \frac{\sum_{i=1}^n (Y_i - \hat{Y}_i)^2}{n - p - 1}$$

### 5.3 Gauss-Markov Theorem (BLUE)

**Theorem:** Under the assumptions of linearity, exogeneity ($E[\epsilon|X] = 0$), homoscedasticity ($\text{Var}(\epsilon|X) = \sigma^2 I$), and no perfect multicollinearity, the OLS estimator is the **Best Linear Unbiased Estimator (BLUE)**.

- **Best:** lowest variance among all linear unbiased estimators
- **Linear:** $\hat{\boldsymbol{\beta}}$ is a linear function of $\mathbf{Y}$
- **Unbiased:** $E[\hat{\boldsymbol{\beta}}] = \boldsymbol{\beta}$

**Important:** Gauss-Markov does NOT require normality of errors. Normality is needed for exact $t$ and $F$ distributions in finite samples, but BLUE holds without it.

**When Gauss-Markov fails:**
- Heteroscedasticity $\Rightarrow$ use White (robust) standard errors or GLS
- Autocorrelation $\Rightarrow$ use Newey-West standard errors or GLS
- Both are common with financial time series data

### 5.4 $R^2$ and Adjusted $R^2$

**Coefficient of Determination:**

$$R^2 = 1 - \frac{\text{RSS}}{\text{TSS}} = 1 - \frac{\sum(Y_i - \hat{Y}_i)^2}{\sum(Y_i - \bar{Y})^2}$$

$R^2 \in [0, 1]$ measures the fraction of variance in $Y$ explained by the model. In simple regression, $R^2 = r_{XY}^2$ (squared correlation).

**Problem:** $R^2$ never decreases when adding predictors, even irrelevant ones.

**Adjusted $R^2$** penalizes for the number of predictors:

$$\bar{R}^2 = 1 - \frac{\text{RSS}/(n-p-1)}{\text{TSS}/(n-1)} = 1 - \frac{n-1}{n-p-1}(1-R^2)$$

$\bar{R}^2$ can decrease if adding a predictor does not improve the fit enough to offset the lost degree of freedom.

> **Interview Tip:** In finance, low $R^2$ does not mean a model is useless. Daily stock return regressions typically have $R^2 < 0.05$, but the economic significance can be enormous. An $R^2$ of 0.01 for daily returns corresponds to a Sharpe ratio of about 1.0 annualized, which is exceptional.

### 5.5 Standard Errors, $t$-Statistics, and Multicollinearity

The standard error of $\hat{\beta}_j$ is:

$$\text{SE}(\hat{\beta}_j) = \hat{\sigma} \sqrt{[(\mathbf{X}'\mathbf{X})^{-1}]_{jj}}$$

The $t$-statistic for testing $H_0: \beta_j = 0$:

$$t_j = \frac{\hat{\beta}_j}{\text{SE}(\hat{\beta}_j)} \sim t_{n-p-1}$$

**Multicollinearity** occurs when predictors are highly correlated. Effects:
- $(\mathbf{X}'\mathbf{X})$ is nearly singular $\Rightarrow$ large diagonal elements in $(\mathbf{X}'\mathbf{X})^{-1}$
- Standard errors inflate $\Rightarrow$ individual coefficients appear insignificant
- Coefficient estimates become unstable (small data changes $\Rightarrow$ large $\hat{\beta}$ changes)

**Variance Inflation Factor (VIF):**

$$\text{VIF}_j = \frac{1}{1 - R_j^2}$$

where $R_j^2$ is the $R^2$ from regressing $X_j$ on all other predictors. VIF $> 10$ is a common rule of thumb for problematic multicollinearity.

**Remedies:** Principal Component Regression (PCR), Ridge regression (adds $L_2$ penalty), or dropping redundant variables.

> **Interview Tip:** In multi-factor models (Fama-French), the factors are constructed to be somewhat uncorrelated, but multicollinearity can still arise. If asked about it, mention VIF and Ridge regression as solutions.

---

## 6. Bayesian Statistics

Bayesian statistics provides an alternative framework where parameters are treated as random variables with probability distributions. This is increasingly important in quantitative finance for portfolio construction, signal combination, and risk management.

### 6.1 Bayes' Theorem for Parameters

**Bayes' Theorem:**

$$\pi(\theta | \mathbf{x}) = \frac{f(\mathbf{x}|\theta)\pi(\theta)}{f(\mathbf{x})} \propto f(\mathbf{x}|\theta)\pi(\theta)$$

In words:

$$\text{Posterior} \propto \text{Likelihood} \times \text{Prior}$$

- **Prior** $\pi(\theta)$: our belief about $\theta$ before seeing data
- **Likelihood** $f(\mathbf{x}|\theta)$: the probability of the data given $\theta$
- **Posterior** $\pi(\theta|\mathbf{x})$: updated belief after seeing data
- **Marginal likelihood** $f(\mathbf{x}) = \int f(\mathbf{x}|\theta)\pi(\theta)d\theta$ (normalizing constant)

### 6.2 Conjugate Priors

A **conjugate prior** is one where the posterior belongs to the same family as the prior. This makes computation tractable.

| Likelihood | Conjugate Prior | Posterior |
|-----------|----------------|----------|
| $\text{Normal}(\mu, \sigma^2_{\text{known}})$ | $\mu \sim N(\mu_0, \tau_0^2)$ | $\mu|\mathbf{x} \sim N(\mu_n, \tau_n^2)$ |
| $\text{Bernoulli}(p)$ | $p \sim \text{Beta}(a, b)$ | $p|\mathbf{x} \sim \text{Beta}(a + \sum x_i, b + n - \sum x_i)$ |
| $\text{Poisson}(\lambda)$ | $\lambda \sim \text{Gamma}(a, b)$ | $\lambda|\mathbf{x} \sim \text{Gamma}(a + \sum x_i, b + n)$ |
| $\text{Exponential}(\lambda)$ | $\lambda \sim \text{Gamma}(a, b)$ | $\lambda|\mathbf{x} \sim \text{Gamma}(a + n, b + \sum x_i)$ |

**Normal-Normal Conjugate (most important for finance):**

With prior $\mu \sim N(\mu_0, \tau_0^2)$ and data $X_1, \ldots, X_n \sim N(\mu, \sigma^2)$ (known $\sigma^2$):

$$\mu | \mathbf{x} \sim N(\mu_n, \tau_n^2)$$

where:

$$\mu_n = \frac{\frac{\mu_0}{\tau_0^2} + \frac{n\bar{x}}{\sigma^2}}{\frac{1}{\tau_0^2} + \frac{n}{\sigma^2}}, \quad \frac{1}{\tau_n^2} = \frac{1}{\tau_0^2} + \frac{n}{\sigma^2}$$

The posterior mean $\mu_n$ is a **precision-weighted average** of the prior mean and the sample mean. As $n \to \infty$, $\mu_n \to \bar{x}$ (data overwhelms the prior).

### 6.3 MAP Estimation vs Posterior Mean

Two common point estimates from the posterior:

**Maximum A Posteriori (MAP):**

$$\hat{\theta}_{\text{MAP}} = \arg\max_{\theta} \pi(\theta | \mathbf{x}) = \arg\max_{\theta} [\ln f(\mathbf{x}|\theta) + \ln \pi(\theta)]$$

This is the mode of the posterior. With a flat (uninformative) prior, $\hat{\theta}_{\text{MAP}} = \hat{\theta}_{\text{MLE}}$.

**Posterior Mean:**

$$\hat{\theta}_{\text{PM}} = E[\theta | \mathbf{x}] = \int \theta \, \pi(\theta | \mathbf{x}) \, d\theta$$

The posterior mean minimizes the expected squared loss (Bayes risk under squared loss).

For symmetric posteriors (e.g., normal), MAP $=$ posterior mean. They differ for skewed posteriors.

**Connection to regularization:** MAP estimation with a $N(0, \tau^2)$ prior on regression coefficients is equivalent to **Ridge regression** with penalty $\lambda = \sigma^2/\tau^2$. A Laplace prior gives **LASSO**.

### 6.4 Bayesian vs Frequentist: Comparison

| Aspect | Frequentist | Bayesian |
|--------|------------|----------|
| Parameters | Fixed, unknown constants | Random variables with distributions |
| Probability | Long-run frequency | Degree of belief |
| Inference | Based on sampling distribution of estimators | Based on posterior distribution |
| Prior information | Not formally used | Encoded in the prior |
| Confidence intervals | "95% of intervals from repeated samples contain $\theta$" | "95% probability that $\theta$ is in this interval" |
| Small samples | May rely on asymptotic approximations | Naturally handles small samples via the prior |

**Finance applications of Bayesian methods:**
- **Black-Litterman model:** combines market equilibrium (prior) with investor views (likelihood)
- **Bayesian portfolio optimization:** addresses estimation error by shrinking toward a prior
- **Signal combination:** naturally combines multiple alpha signals with different precisions
- **Parameter uncertainty:** Bayesian predictive distributions integrate over parameter uncertainty rather than plugging in point estimates

> **Interview Tip:** When asked to compare Bayesian and frequentist approaches, emphasize the practical implications: Bayesian methods naturally handle parameter uncertainty and can incorporate prior knowledge (like economic theory). In finance, this is valuable because sample sizes are small and estimation error is a first-order problem.

---

## 7. Time Series Basics for Quantitative Finance

Financial data are inherently temporal. Understanding time series is essential for modeling returns, volatility, and signal dynamics.

### 7.1 Stationarity

**Strict stationarity:** The joint distribution of $(X_{t_1}, \ldots, X_{t_k})$ is the same as $(X_{t_1+h}, \ldots, X_{t_k+h})$ for all $h, k, t_1, \ldots, t_k$.

**Weak (covariance) stationarity** requires only:
1. $E[X_t] = \mu$ (constant mean)
2. $\text{Var}(X_t) = \gamma(0) < \infty$ (constant, finite variance)
3. $\text{Cov}(X_t, X_{t+h}) = \gamma(h)$ depends only on lag $h$, not on $t$

**Why it matters in finance:** Most statistical methods assume stationarity. Stock *prices* are non-stationary (they trend), but *returns* $r_t = \ln(S_t/S_{t-1})$ are approximately stationary. This is why we model returns, not prices.

**Testing for stationarity:** The **Augmented Dickey-Fuller (ADF) test** tests $H_0$: unit root (non-stationary) vs $H_1$: stationary.

### 7.2 Autocorrelation Function (ACF)

The **autocovariance function** at lag $h$:

$$\gamma(h) = \text{Cov}(X_t, X_{t+h}) = E[(X_t - \mu)(X_{t+h} - \mu)]$$

The **autocorrelation function (ACF):**

$$\rho(h) = \frac{\gamma(h)}{\gamma(0)} = \text{Corr}(X_t, X_{t+h})$$

Properties: $\rho(0) = 1$, $|\rho(h)| \leq 1$, $\rho(h) = \rho(-h)$.

**Sample ACF:**

$$\hat{\rho}(h) = \frac{\sum_{t=1}^{n-h}(x_t - \bar{x})(x_{t+h} - \bar{x})}{\sum_{t=1}^{n}(x_t - \bar{x})^2}$$

Under the null of no autocorrelation, $\hat{\rho}(h)$ is approximately $N(0, 1/n)$ for large $n$. The 95% confidence bands are $\pm 1.96/\sqrt{n}$.

**Ljung-Box test** for joint significance of multiple lags:

$$Q = n(n+2)\sum_{h=1}^{m}\frac{\hat{\rho}(h)^2}{n-h} \sim \chi^2_m \text{ under } H_0$$

### 7.3 AR(1) Process

$$X_t = \phi X_{t-1} + \epsilon_t, \quad \epsilon_t \sim \text{WN}(0, \sigma^2)$$

**Stationarity condition:** $|\phi| < 1$

For a stationary AR(1):
- $E[X_t] = 0$ (assuming zero mean)
- $\text{Var}(X_t) = \frac{\sigma^2}{1 - \phi^2}$
- $\rho(h) = \phi^h$ (ACF decays exponentially)

The **half-life** of the process (time for ACF to decay by half):

$$h_{1/2} = -\frac{\ln 2}{\ln |\phi|}$$

For $\phi = 0.9$: half-life $\approx 6.6$ periods. For $\phi = 0.99$: half-life $\approx 69$ periods.

### 7.4 MA(1) Process

$$X_t = \epsilon_t + \theta \epsilon_{t-1}, \quad \epsilon_t \sim \text{WN}(0, \sigma^2)$$

Properties:
- Always stationary (for any $\theta$)
- $E[X_t] = 0$
- $\text{Var}(X_t) = (1 + \theta^2)\sigma^2$
- ACF: $\rho(1) = \frac{\theta}{1 + \theta^2}$, $\rho(h) = 0$ for $h \geq 2$

The ACF **cuts off** after lag 1 --- this is the signature of an MA(1) process.

### 7.5 Random Walk

$$X_t = X_{t-1} + \epsilon_t, \quad \epsilon_t \sim \text{WN}(0, \sigma^2)$$

This is an AR(1) with $\phi = 1$ (unit root) --- it is **not stationary**.

Properties:
- $E[X_t] = X_0$
- $\text{Var}(X_t) = t\sigma^2$ (variance grows linearly with time)
- $X_t = X_0 + \sum_{i=1}^t \epsilon_i$ (sum of all past shocks)

**Random walk with drift:** $X_t = \mu + X_{t-1} + \epsilon_t$. Now $E[X_t] = X_0 + \mu t$ (linear trend).

**The Efficient Market Hypothesis (weak form)** implies that log-prices follow approximately a random walk: $\ln S_t = \ln S_{t-1} + \mu + \epsilon_t$. If returns were predictable ($\phi \neq 0$ in an AR model for returns), you could profit from it.

> **Interview Tip:** Be able to explain the difference between "stock prices follow a random walk" and "stock returns are unpredictable." They are essentially the same statement. Prices are non-stationary (random walk), but returns (first differences of log-prices) are stationary and approximately iid.

### 7.6 Mean Reversion and the Ornstein-Uhlenbeck Process

Many financial quantities exhibit **mean reversion**: they tend to return toward a long-run mean. Examples include interest rates, volatility, and pairs trading spreads.

The **discrete-time mean-reverting (AR(1)) process:**

$$X_t - \mu = \phi(X_{t-1} - \mu) + \epsilon_t, \quad |\phi| < 1$$

Equivalently: $X_t = (1-\phi)\mu + \phi X_{t-1} + \epsilon_t$

The **continuous-time analog** is the **Ornstein-Uhlenbeck (OU) process:**

$$dX_t = \kappa(\mu - X_t)dt + \sigma dW_t$$

where:
- $\kappa > 0$: speed of mean reversion
- $\mu$: long-run mean
- $\sigma$: volatility of the noise
- $W_t$: standard Brownian motion

**Solution:**

$$X_t = \mu + (X_0 - \mu)e^{-\kappa t} + \sigma \int_0^t e^{-\kappa(t-s)} dW_s$$

**Stationary distribution:** $X_t \sim N\left(\mu, \frac{\sigma^2}{2\kappa}\right)$ as $t \to \infty$.

**Half-life:** $h_{1/2} = \frac{\ln 2}{\kappa}$

**Relationship to discrete AR(1):** With time step $\Delta t$, $\phi = e^{-\kappa \Delta t}$.

> **Interview Tip:** The OU process is central to pairs trading and interest rate models (Vasicek model). A typical interview question: "How would you determine if a spread is mean-reverting, and how would you estimate the half-life?" Answer: test for stationarity (ADF test), fit an AR(1), and compute $h_{1/2} = -\ln 2 / \ln \hat{\phi}$.

---

## 8. Statistical Distributions in Finance

The choice of distributional model matters enormously in finance, because tail behavior drives risk and hedging decisions.

### 8.1 The Log-Normal Distribution (Stock Prices)

If log-returns are normally distributed, then prices are **log-normally distributed**.

Under geometric Brownian motion, $dS_t = \mu S_t dt + \sigma S_t dW_t$, the solution is:

$$S_T = S_0 \exp\left[\left(\mu - \frac{\sigma^2}{2}\right)T + \sigma W_T\right]$$

Therefore:

$$\ln S_T \sim N\left(\ln S_0 + \left(\mu - \frac{\sigma^2}{2}\right)T, \; \sigma^2 T\right)$$

**Properties of the log-normal:**

If $Y = \ln X \sim N(m, v^2)$, then:
- $E[X] = e^{m + v^2/2}$
- $\text{Var}(X) = e^{2m + v^2}(e^{v^2} - 1)$
- $\text{Median}(X) = e^m < E[X]$ (right-skewed)

**Key insight:** The $-\sigma^2/2$ term (convexity adjustment, or Ito correction) explains why the expected log-return differs from the log of the expected return. This distinction between **arithmetic** and **geometric** returns is fundamental.

> **Interview Tip:** "If a stock has an expected return of 10% and volatility of 30%, what is the expected log-return?" Answer: $\mu - \sigma^2/2 = 0.10 - 0.09/2 = 0.055 = 5.5\%$. The geometric return is always less than the arithmetic return (except when $\sigma = 0$). This is called **volatility drag**.

### 8.2 Fat Tails and Excess Kurtosis

Empirically, financial returns exhibit **fat tails** (leptokurtosis): extreme events occur far more often than a normal model predicts.

**Stylized facts of financial returns:**
1. Excess kurtosis $\gg 0$ (typically 5--50 for daily returns)
2. Negative skewness (especially for equity indices)
3. Volatility clustering: large moves tend to follow large moves
4. Leverage effect: negative returns increase future volatility

**Quantifying tail risk:** For a standard normal, $P(|Z| > 4) \approx 0.006\%$. For actual daily equity returns, moves of 4 standard deviations occur roughly 10-100 times more often.

**Why normality fails:**

| Event | Normal model probability | Observed frequency |
|-------|------------------------|-------------------|
| 3$\sigma$ move | 0.27% | ~1-2% |
| 4$\sigma$ move | 0.006% | ~0.1-0.5% |
| 5$\sigma$ move | 0.00006% | ~0.01-0.05% |

This gap between model and reality is why risk management must account for fat tails.

### 8.3 Student-$t$ Distribution for Heavy Tails

The **Student-$t$ distribution** with $\nu$ degrees of freedom has density:

$$f(x) = \frac{\Gamma\left(\frac{\nu+1}{2}\right)}{\sqrt{\nu\pi}\;\Gamma\left(\frac{\nu}{2}\right)}\left(1 + \frac{x^2}{\nu}\right)^{-(\nu+1)/2}$$

**Tail behavior:** For large $|x|$, $f(x) \sim |x|^{-(\nu+1)}$ (power-law tails). This is much heavier than the Gaussian $e^{-x^2/2}$.

**Moments:**
- $E[X] = 0$ for $\nu > 1$
- $\text{Var}(X) = \nu/(\nu - 2)$ for $\nu > 2$
- Kurtosis $= 3 + 6/(\nu - 4)$ for $\nu > 4$
- Moments of order $k$ exist only if $\nu > k$

The **location-scale $t$-distribution** $X = \mu + \sigma T_\nu$ is commonly used to model financial returns. Typical fits give $\nu \approx 3$--$8$ for daily equity returns.

> **Interview Tip:** A $t$-distribution with $\nu = 4$ has infinite kurtosis. With $\nu = 3$, even the variance diverges slowly. For daily returns, $\nu \approx 5$ is typical, giving kurtosis of $3 + 6/(5-4) = 9$ (excess kurtosis of 6). This is far from the normal distribution's kurtosis of 3.

### 8.4 Mixture Distributions

A **mixture distribution** combines multiple component distributions:

$$f(x) = \sum_{k=1}^{K} w_k f_k(x), \quad \sum_{k=1}^K w_k = 1, \quad w_k \geq 0$$

**Gaussian Mixture Model (GMM):**

$$f(x) = \sum_{k=1}^K w_k \cdot \frac{1}{\sigma_k\sqrt{2\pi}} \exp\left(-\frac{(x-\mu_k)^2}{2\sigma_k^2}\right)$$

**Example: Two-state model for returns.** Let returns follow:
- Normal market: $N(\mu_1, \sigma_1^2)$ with probability $w$
- Stressed market: $N(\mu_2, \sigma_2^2)$ with probability $1-w$

Even if both components are normal, the mixture is **not** normal. It naturally generates fat tails and excess kurtosis.

**Variance of a mixture:**

$$\text{Var}(X) = \sum_k w_k(\sigma_k^2 + \mu_k^2) - \left(\sum_k w_k \mu_k\right)^2$$

The additional variance from different means is the source of excess kurtosis.

> **Interview Tip:** Mixture models connect to **regime-switching models** in finance (Hamilton, 1989). The idea: markets alternate between calm and volatile regimes, and a mixture of normals captures this. Parameters are estimated via the **EM algorithm**.

---

## 9. Monte Carlo Methods

Monte Carlo simulation uses random sampling to estimate quantities that are difficult or impossible to compute analytically. It is ubiquitous in quantitative finance for pricing derivatives, computing risk measures, and evaluating trading strategies.

### 9.1 Basic Monte Carlo Estimation

**Goal:** Estimate $\theta = E[g(X)]$ where $X \sim f$.

**Algorithm:**
1. Generate iid samples $X_1, \ldots, X_N \sim f$
2. Compute $\hat{\theta}_N = \frac{1}{N}\sum_{i=1}^N g(X_i)$

By the **Law of Large Numbers**, $\hat{\theta}_N \xrightarrow{a.s.} \theta$.

By the **Central Limit Theorem**:

$$\hat{\theta}_N \approx N\left(\theta, \frac{\sigma_g^2}{N}\right) \quad \text{where } \sigma_g^2 = \text{Var}(g(X))$$

**Standard error:**

$$\text{SE} = \frac{\sigma_g}{\sqrt{N}}$$

The convergence rate is $O(1/\sqrt{N})$ regardless of dimension. This means:
- To halve the error, you need $4\times$ more samples
- For one extra decimal digit of accuracy, you need $100\times$ more samples

**Example: Pricing a European call option.**

Under risk-neutral pricing:

$$C = e^{-rT} E[\max(S_T - K, 0)]$$

Simulate $N$ paths of $S_T = S_0 e^{(r - \sigma^2/2)T + \sigma\sqrt{T}Z_i}$, $Z_i \sim N(0,1)$:

$$\hat{C} = e^{-rT} \cdot \frac{1}{N}\sum_{i=1}^N \max(S_0 e^{(r-\sigma^2/2)T + \sigma\sqrt{T}Z_i} - K, 0)$$

### 9.2 Variance Reduction Techniques

Since $\text{SE} = \sigma_g / \sqrt{N}$, we can improve efficiency by reducing $\sigma_g$ (variance reduction) instead of increasing $N$.

#### Antithetic Variables

If $Z \sim N(0,1)$, then $-Z \sim N(0,1)$ as well. Use both $Z_i$ and $-Z_i$ in each pair:

$$\hat{\theta}_{\text{AV}} = \frac{1}{N}\sum_{i=1}^{N/2}\frac{g(Z_i) + g(-Z_i)}{2}$$

This works when $g$ is monotonic (introducing negative correlation between paired estimates). For option pricing, antithetic variates typically reduce variance by 50-80%.

**Variance reduction factor:**

$$\text{Var}(\hat{\theta}_{\text{AV}}) = \frac{1}{N}\left[\text{Var}(g(Z)) + \text{Cov}(g(Z), g(-Z))\right]$$

If $g$ is monotonic, $\text{Cov}(g(Z), g(-Z)) < 0$, reducing variance.

#### Control Variates

Suppose we know $E[h(X)] = \mu_h$ exactly for some function $h$. Define:

$$\hat{\theta}_{\text{CV}} = \hat{\theta}_N - c\left(\frac{1}{N}\sum_{i=1}^N h(X_i) - \mu_h\right)$$

The optimal $c$ is:

$$c^* = \frac{\text{Cov}(g(X), h(X))}{\text{Var}(h(X))}$$

**Variance reduction:**

$$\text{Var}(\hat{\theta}_{\text{CV}}) = (1 - \rho^2_{gh})\text{Var}(\hat{\theta}_N)$$

where $\rho_{gh}$ is the correlation between $g(X)$ and $h(X)$.

**Example:** When pricing an exotic option, use the Black-Scholes price of a vanilla option as a control variate. If the exotic payoff is highly correlated with the vanilla payoff, this can reduce variance by orders of magnitude.

### 9.3 Importance Sampling

**Idea:** Sample from a different distribution $q$ that places more probability where the integrand is large.

$$\theta = E_f[g(X)] = \int g(x)f(x)dx = \int g(x)\frac{f(x)}{q(x)}q(x)dx = E_q\left[g(X)\frac{f(X)}{q(X)}\right]$$

**Importance sampling estimator:**

$$\hat{\theta}_{\text{IS}} = \frac{1}{N}\sum_{i=1}^N g(X_i)\frac{f(X_i)}{q(X_i)}, \quad X_i \sim q$$

The ratio $w(x) = f(x)/q(x)$ is the **importance weight**.

**Optimal proposal:** $q^*(x) \propto |g(x)|f(x)$, which gives zero variance. In practice, we use approximations.

**Finance application:** Estimating the probability of rare events (e.g., a portfolio losing more than 10%). Under the original distribution, very few simulations reach the tail. Importance sampling shifts the distribution toward the tail, dramatically reducing variance.

> **Interview Tip:** "How would you efficiently estimate the 99.9th percentile VaR using Monte Carlo?" Standard MC needs millions of samples for the 99.9th percentile. Importance sampling can achieve the same accuracy with orders of magnitude fewer samples by sampling from a distribution centered near the tail.

---

## 10. Classic Interview Problems

This section presents worked examples of the type commonly encountered in quant finance interviews.

### Problem 1: MLE for the Exponential Distribution

**Problem:** You observe waiting times $x_1, \ldots, x_n$ between trades, modeled as iid $\text{Exponential}(\lambda)$ with density $f(x|\lambda) = \lambda e^{-\lambda x}$ for $x > 0$. Find the MLE of $\lambda$ and its asymptotic variance.

**Solution:**

**Step 1: Write the log-likelihood.**

$$\ell(\lambda) = \sum_{i=1}^n \ln(\lambda e^{-\lambda x_i}) = n\ln\lambda - \lambda\sum_{i=1}^n x_i$$

**Step 2: Take the derivative and set to zero.**

$$\frac{d\ell}{d\lambda} = \frac{n}{\lambda} - \sum_{i=1}^n x_i = 0$$

$$\hat{\lambda}_{\text{MLE}} = \frac{n}{\sum_{i=1}^n x_i} = \frac{1}{\bar{x}}$$

**Step 3: Verify it is a maximum.**

$$\frac{d^2\ell}{d\lambda^2} = -\frac{n}{\lambda^2} < 0 \quad \checkmark$$

**Step 4: Find the Fisher information.**

$$I(\lambda) = -E\left[\frac{d^2\ell}{d\lambda^2}\right] = \frac{n}{\lambda^2} \quad \Rightarrow \quad I_1(\lambda) = \frac{1}{\lambda^2}$$

**Step 5: Asymptotic variance.**

$$\text{Var}(\hat{\lambda}) \approx \frac{1}{nI_1(\lambda)} = \frac{\lambda^2}{n}$$

So $\hat{\lambda} \approx N(\lambda, \lambda^2/n)$ for large $n$.

> **Interview Tip:** Notice that $\hat{\lambda}$ achieves the CRLB, so it is asymptotically efficient. Also note: $E[1/\bar{X}] \neq 1/E[\bar{X}] = \lambda$ by Jensen's inequality, so the MLE is actually biased in finite samples (but consistent).

### Problem 2: Hypothesis Test for a Trading Strategy

**Problem:** A trading strategy produces daily returns over 252 trading days. The sample mean return is $\bar{r} = 0.05\%$ per day and the sample standard deviation is $s = 1.2\%$ per day. Test whether the strategy has a statistically significant positive mean return at the 5% level.

**Solution:**

**Step 1: Set up the hypotheses.**

$$H_0: \mu \leq 0 \quad \text{vs} \quad H_1: \mu > 0$$

**Step 2: Compute the test statistic.**

$$t = \frac{\bar{r} - 0}{s/\sqrt{n}} = \frac{0.0005}{0.012/\sqrt{252}} = \frac{0.0005}{0.000756} \approx 0.661$$

**Step 3: Find the critical value.**

For a one-sided test at $\alpha = 0.05$ with $\nu = 251$ df: $t_{251, 0.05} \approx 1.651$.

**Step 4: Decision.**

Since $0.661 < 1.651$, we **fail to reject** $H_0$. There is insufficient evidence that the strategy has a positive mean return.

**Step 5: Annualized Sharpe ratio perspective.**

Daily Sharpe ratio: $\bar{r}/s = 0.05/1.2 \approx 0.042$

Annualized: $0.042 \times \sqrt{252} \approx 0.66$

Note: The $t$-statistic equals the daily Sharpe ratio times $\sqrt{n}$:

$$t = \frac{\bar{r}}{s/\sqrt{n}} = \text{SR}_{\text{daily}} \times \sqrt{n} \approx \text{SR}_{\text{annual}}$$

(since $n = 252$ trading days $\approx$ 1 year).

> **Interview Tip:** To achieve statistical significance ($t > 1.96$ for a two-sided test), you roughly need an annualized Sharpe ratio above 2.0 with one year of data. For a Sharpe of 0.66, you would need approximately $n = (1.96/0.042)^2 \approx 2177$ trading days (about 8.6 years) to achieve significance at 5%.

### Problem 3: Bayesian Updating of a Return Estimate

**Problem:** Your prior belief about a fund's annual excess return $\mu$ is $N(0, (3\%)^2)$ (centered at zero with standard deviation 3%). You observe 4 years of returns with a sample mean of 2% per year and a known annual standard deviation of $\sigma = 10\%$. What is your posterior estimate of $\mu$?

**Solution:**

**Step 1: Identify the components.**

- Prior: $\mu \sim N(\mu_0, \tau_0^2) = N(0, 0.03^2)$
- Data: $\bar{x} = 0.02$, $\sigma = 0.10$, $n = 4$
- Prior precision: $1/\tau_0^2 = 1/0.0009 \approx 1111$
- Data precision: $n/\sigma^2 = 4/0.01 = 400$

**Step 2: Compute the posterior.**

$$\frac{1}{\tau_n^2} = \frac{1}{\tau_0^2} + \frac{n}{\sigma^2} = 1111 + 400 = 1511$$

$$\tau_n^2 = \frac{1}{1511} \approx 0.000662, \quad \tau_n \approx 2.57\%$$

$$\mu_n = \tau_n^2 \left(\frac{\mu_0}{\tau_0^2} + \frac{n\bar{x}}{\sigma^2}\right) = \frac{1}{1511}\left(\frac{0}{0.0009} + \frac{4 \times 0.02}{0.01}\right) = \frac{8}{1511} \approx 0.53\%$$

**Step 3: Interpret.**

$$\mu | \text{data} \sim N(0.53\%, 2.57\%^2)$$

Despite observing a 2% annual excess return, the posterior mean is only 0.53%. The prior (skepticism about alpha) dominates because:
- The prior is quite informative (3% std dev)
- Only 4 years of data provides weak evidence ($\sigma/\sqrt{n} = 10\%/\sqrt{4} = 5\%$)

The posterior is a precision-weighted average:

$$\mu_n = \underbrace{\frac{1111}{1511}}_{\approx 0.74} \times 0\% + \underbrace{\frac{400}{1511}}_{\approx 0.26} \times 2\% = 0.53\%$$

The prior gets about 74% weight, the data only 26%.

> **Interview Tip:** This problem illustrates **shrinkage** --- a central concept in portfolio management. With noisy estimates and strong priors, the Bayesian posterior shrinks the MLE toward the prior mean. This is exactly what the Black-Litterman model does: it shrinks expected return estimates toward equilibrium, preventing extreme portfolio weights caused by estimation error.

### Problem 4: Interpreting a Regression Output

**Problem:** You run the following CAPM regression on monthly data for a hedge fund (60 months):

$$R_{\text{fund},t} - R_{f,t} = \hat{\alpha} + \hat{\beta}(R_{M,t} - R_{f,t}) + \hat{\epsilon}_t$$

**Results:**

| Coefficient | Estimate | Std Error | $t$-statistic | $p$-value |
|:-----------:|:--------:|:---------:|:------------:|:---------:|
| $\alpha$ | 0.40% | 0.18% | 2.22 | 0.030 |
| $\beta$ | 0.65 | 0.12 | 5.42 | $< 0.001$ |

$R^2 = 0.34$, $n = 60$ months.

Interpret these results.

**Solution:**

**Alpha:** $\hat{\alpha} = 0.40\%$ per month $\approx 4.8\%$ annualized. The $t$-statistic of 2.22 exceeds the 5% critical value ($t_{58, 0.025} \approx 2.00$), so $\alpha$ is statistically significant at the 5% level ($p = 0.030$). The fund appears to generate positive risk-adjusted returns.

**Beta:** $\hat{\beta} = 0.65$ means the fund has below-market systematic risk exposure. A 1% market move corresponds to approximately a 0.65% fund move. Highly significant ($p < 0.001$).

**$R^2 = 0.34$:** The market explains 34% of the variation in fund returns. The remaining 66% is idiosyncratic. This is typical for a hedge fund --- higher than a market-neutral fund but lower than a long-only equity fund.

**Caveats to mention in an interview:**
1. **Non-normal returns:** Hedge funds often have non-normal return distributions (short vol, illiquidity), making $t$-tests unreliable.
2. **Survivorship bias:** Funds that failed are not in the sample.
3. **Multiple testing:** If you screened many funds, the significant alpha may be a false positive.
4. **Heteroscedasticity:** Standard errors may be biased. Use robust (White) or Newey-West standard errors.
5. **Missing factors:** A Fama-French or multi-factor model might explain away the alpha.

> **Interview Tip:** Always discuss economic significance alongside statistical significance. An annualized alpha of 4.8% is economically meaningful. But also compute the **information ratio** $= \alpha / \sigma_{\epsilon} \approx 0.40\% / \sqrt{\text{Var}(\epsilon)}$. If the residual vol is about 2.5% per month, the monthly IR $\approx 0.16$, annualized IR $\approx 0.55$, which is a solid risk-adjusted performance.

---

## Key Formulas Reference Card

A quick-reference summary of the most important formulas for interview preparation.

| Topic | Formula |
|-------|--------|
| Sample variance | $s^2 = \frac{1}{n-1}\sum(x_i - \bar{x})^2$ |
| MLE (general) | $\hat{\theta} = \arg\max \sum \ln f(x_i \mid \theta)$ |
| Fisher information | $I(\theta) = -E[\partial^2 \ln f / \partial\theta^2]$ |
| Cramer-Rao bound | $\text{Var}(\hat{\theta}) \geq 1/(nI(\theta))$ |
| $t$-statistic | $t = (\bar{x} - \mu_0)/(s/\sqrt{n})$ |
| OLS estimator | $\hat{\beta} = (X'X)^{-1}X'Y$ |
| Bayes' theorem | $\pi(\theta \mid x) \propto f(x\mid\theta)\pi(\theta)$ |
| AR(1) stationarity | $|\phi| < 1$, $\text{Var} = \sigma^2/(1-\phi^2)$ |
| OU half-life | $h_{1/2} = \ln 2 / \kappa$ |
| Log-normal stock | $\ln S_T \sim N(\ln S_0 + (\mu - \sigma^2/2)T, \sigma^2 T)$ |
| MC standard error | $\text{SE} = \sigma / \sqrt{N}$ |
| Sharpe $\leftrightarrow$ $t$-stat | $t \approx \text{SR} \times \sqrt{n}$ |

---

*This notebook is part of the Quant Finance Interview Prep series. Master these concepts and you will be well prepared for the statistical inference portion of any quantitative finance interview.*